# Bundled Exasol lesson: Flight Utils

Downloads the tutorial dataset when you run the loader. Tutorial tables are kept in EXAKIT_EXAMPLES, separately from the persona datasets.

Run cells from top to bottom. In Marimo each editable cell has its own Run button; opening a lesson executes no example code. Widget steps retain their Python code; use the matching Jupyter notebook for the original widget UI.

For dashboards use the kit Data Scientist 03_share flow on Dash Server.

Source: [Exasol Notebook Connector](https://github.com/exasol/notebook-connector/blob/14e2c7ad088b7109aa21d283e98f3126bbbe27e5/exasol/nb_connector/resources/notebooks/data/utils/flight_utils.ipynb). MIT license: LICENSE.txt.

In [ ]:
from exakit.nb.example_runtime import bootstrap
ai_lab_config = bootstrap(globals(), 'data/utils/flight_utils.ipynb')

<div style="text-align: right;">
  <img src="https://raw.githubusercontent.com/exasol/ai-lab/refs/heads/main/assets/Exasol_Logo_2025_Dark.svg" style="width:200px; margin: 10px;" />
</div>

In [ ]:
from exasol.nb_connector.utils import upward_file_search

# This NB may be running from various locations in the NB hierarchy.
# Need to search for other supporting NBs from the current directory upwards.

from exasol.nb_connector.ui.common import ui_styles
from exasol.nb_connector.ui.common import popup_message

In [ ]:
from typing import List
from datetime import datetime
from dateutil.relativedelta import relativedelta
import io
from itertools import islice

import ipywidgets as widgets
import requests
from exasol.nb_connector.secret_store import Secrets
from exakit.nb.example_runtime import open_pyexasol_connection


def _transform_flights(pipe, src) -> None:

    for row in islice(src, 1, None):
        fields = row.split(',')
        fields[0] = datetime.strptime(fields[0], '%m/%d/%Y %I:%M:%S %p').strftime('%Y-%m-%d')
        for i in [10, 12]:
            fields[i] = 'False' if float(fields[i]) == 0. else 'True'
        pipe.write(bytes(','.join(fields), encoding='utf-8'))


def _transform_airlines(pipe, src) -> None:

    for row in islice(src, 1, None):
        fields = row.split(',', maxsplit=1)
        al_name = fields[1].strip('"\n').split(':')[0]
        fields[1] = f'"{al_name}"'
        pipe.write(bytes(','.join(fields) + '\n', encoding='utf-8'))


def _import_from_cloudfront(conf: Secrets, file_name: str, transform, table_name: str) -> None:

    # Read requested csv file from the cloudfront
    response = requests.get(f'https://dut5tonqye28.cloudfront.net/ai_lab/flight-info/{file_name}')
    response.raise_for_status()
    content_stream = io.BytesIO(response.content)
    f_src = io.TextIOWrapper(content_stream, encoding='utf-8')

    # Pre-process and import the data
    with open_pyexasol_connection(conf, schema=conf.db_schema, compression=True) as pyexasol_conn:
        pyexasol_conn.import_from_callback(transform, f_src, table=table_name)


def load_flights_data(conf: Secrets, months: List[str]) -> None:

    for mon in months:
        file_name = f'US_FLIGHTS_{mon.replace(" ", "_").upper()}.csv'
        _import_from_cloudfront(conf, file_name, _transform_flights, 'US_FLIGHTS')


def load_airlines_data(conf: Secrets) -> None:

    _import_from_cloudfront(conf, 'US_AIRLINES.csv', _transform_airlines, 'US_AIRLINES')


def get_data_selection_ui(conf: Secrets) -> widgets.Widget:
    """
    Builds a UI with a multi-select list of data periods and buttons for
    loading the selected data.
    """

    ui_look = ui_styles.config_styles()

    start_date = datetime(year=2023, month=4, day=1)
    months = [(start_date + relativedelta(months=i)).strftime('%b %Y')
              for i in range(12)]

    data_selector = widgets.SelectMultiple(options=months, layout=ui_look.input_layout, style=ui_look.input_style)
    flights_btn = widgets.Button(description='Load Flights', style=ui_look.button_style, layout=ui_look.button_layout)
    airlines_btn = widgets.Button(description='Load Airlines', style=ui_look.button_style, layout=ui_look.button_layout)
    header_lbl = widgets.Label(value='Data Periods', style=ui_look.header_style, layout=ui_look.header_layout)

    def load_flights(btn):
        if data_selector.value:
            try:
                load_flights_data(conf, data_selector.value)
                popup_message.display_popup('Flights data has been loaded successfully')
                data_selector.options = [opt for opt in data_selector.options 
                                         if opt not in data_selector.value]
                btn.icon = 'check'
                btn.disabled = not data_selector.options
            except Exception as ex:
                popup_message.display_popup(f'Failed to load the flights data: {ex}')
        else:
            btn.icon = 'check'

    def load_airlines(btn):
        try:
            load_airlines_data(conf)
            popup_message.display_popup('Airlines data has been loaded successfully')
            btn.disabled = True
        except Exception as ex:
            popup_message.display_popup(f'Failed to load the airlines data: {ex}')

    def on_value_change(change):
        flights_btn.icon = 'pen'

    flights_btn.on_click(load_flights)
    airlines_btn.on_click(load_airlines)
    data_selector.observe(on_value_change, names=['value'])

    group_items = [header_lbl, widgets.Box([data_selector], layout=ui_look.row_layout)]
    items = [widgets.Box(group_items, layout=ui_look.group_layout), 
             widgets.Box([flights_btn, airlines_btn], layout=ui_look.row_layout)]
    ui = widgets.Box(items, layout=ui_look.outer_layout)
    return ui